# AI-Powered Flood Risk Prediction & Geospatial Intelligence System
### Study Area: Lokoja Confluence & Lower Niger Basin, Kogi State, Nigeria
### Target Reference Event: October 2022 Nigeria Flood (Peak Inundation Composite, 1-25 Oct 2022)
### Flood Sensor Ground Truth: VIIRS (Suomi NPP & NOAA-20), UNOSAT Activation FL20221019NGA (Product 3427)
### Capstone Project — TechCrush C8

---
This notebook implements an end-to-end, reproducible, scientifically defensible flood susceptibility modeling pipeline:
1. **Authoritative Geospatial Data Ingestion**: SRTM 30m DEM, CHIRPS rainfall, HydroRIVERS hydrography, ESA WorldCover, UNOSAT satellite flood extent (VIIRS sensor, FL20221019NGA)
2. **Spatial Preprocessing**: Alignment to a common 90m UTM Zone 32N metric grid (EPSG:32632)
3. **Feature Engineering**: Elevation, slope, rainfall accumulation, Euclidean distance to river, soil permeability index
4. **Data Leakage Prevention**: 2x2 Spatial block split (quadrant-based evaluation with domain-shift audit)
5. **Baseline Benchmark & Modelling**: Naive Majority Baseline, Balanced Logistic Regression, and Balanced Random Forest
6. **Model Evaluation & Geospatial Intelligence**: ROC-AUC, PR-AUC (Precision-Recall curves), Confusion Matrices, Continuous Flood Susceptibility Index (FSI), 5-Tier Susceptibility Map, Interactive Folium Map
7. **Decision Log & Limitations**: Defensible, transparent reporting of all assumptions, domain shift findings, and trade-offs.



In [ ]:
# Install required geospatial & ML libraries (Google Colab / Jupyter)
!pip install -q rasterio geopandas pyproj pyogrio folium scikit-learn matplotlib seaborn requests shapely scipy


In [ ]:
"""
================================================================================
AI-POWERED FLOOD RISK PREDICTION & GEOSPATIAL INTELLIGENCE SYSTEM
================================================================================
Study Area  : Lokoja Confluence & Lower Niger Basin, Kogi State, Nigeria
Target Event: October 2022 Nigeria Flood (Peak Inundation Composite, 1-25 Oct 2022)
ML Problem  : Binary Spatial Classification → Continuous Flood Susceptibility Index
Flood Sensor: VIIRS (Suomi NPP & NOAA-20), UNOSAT Activation FL20221019NGA (Product 3427)
Authors     : TechCrush C8 Capstone Team
Reproducibility: RANDOM_STATE = 42 throughout. All data from authoritative sources.

USAGE — GOOGLE COLAB:
  1. Upload this file to Google Colab
  2. Run: exec(open('flood_risk_notebook.py').read())
  OR paste each section into separate notebook cells
  
USAGE — LOCAL:
  python flood_risk_notebook.py
================================================================================
"""


## Stage 1: ENVIRONMENT SETUP


In [ ]:
print("\n" + "="*70)
print("STAGE 1 — ENVIRONMENT SETUP")
print("="*70)

import subprocess, sys, os

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

PACKAGES = [
    'rasterio', 'geopandas', 'pyproj', 'pyogrio',
    'folium', 'scikit-learn', 'matplotlib', 'seaborn',
    'requests', 'shapely',
]

# Quick import check to avoid slow redundant pip installs
missing = []
for pkg, mod in [('rasterio', 'rasterio'), ('geopandas', 'geopandas'), ('pyproj', 'pyproj'),
                 ('folium', 'folium'), ('scikit-learn', 'sklearn'), ('matplotlib', 'matplotlib')]:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing packages: {missing}...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=False)

import io, zipfile, gzip, shutil, json, warnings
import numpy as np
import pandas as pd
import matplotlib
# Prevent GUI hang when run outside interactive notebook
if 'get_ipython' not in globals():
    try:
        matplotlib.use('Agg')
    except Exception:
        pass
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
import requests

import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize

import geopandas as gpd
from shapely.geometry import box
from pyproj import Transformer

from scipy.ndimage import distance_transform_edt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    average_precision_score, precision_recall_curve
)
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

import folium
from folium.plugins import HeatMap

warnings.filterwarnings('ignore')
np.random.seed(42)

print("All imports successful.")
print(f"  rasterio  : {rasterio.__version__}")
print(f"  geopandas : {gpd.__version__}")
print(f"  numpy     : {np.__version__}")


## Stage 2: PROJECT CONFIGURATION


In [ ]:
print("\n" + "="*70)
print("STAGE 2 — PROJECT CONFIGURATION")
print("="*70)

# ── Geographic extent — Lokoja Confluence, Kogi State, Nigeria ─────────────
# Bounding box in WGS84 (EPSG:4326)
BBOX_WEST  =  6.40   # longitude west
BBOX_EAST  =  7.20   # longitude east
BBOX_SOUTH =  7.40   # latitude south
BBOX_NORTH =  8.20   # latitude north

# Coordinate systems
CRS_GEO  = 'EPSG:4326'    # geographic (degrees) — source CRS for most datasets
CRS_PROJ = 'EPSG:32632'   # UTM Zone 32N (metres) — master analysis CRS

# Master grid resolution (metres) — all rasters aligned to this
RESOLUTION_M = 90

# Reproducibility
RANDOM_STATE = 42

# ── Directory structure ────────────────────────────────────────────────────
# Flexible Base Directory: works both in Google Colab and local environments
import os
if os.path.exists('/content') and not os.path.exists('./data'):
    BASE_DIR  = '/content/flood_risk'
else:
    BASE_DIR  = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

DATA_DIR  = os.path.join(BASE_DIR, 'data')
OUT_DIR   = os.path.join(BASE_DIR, 'outputs')
MAP_DIR   = os.path.join(OUT_DIR, 'maps')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
MODEL_DIR = os.path.join(OUT_DIR, 'models')

for d in [DATA_DIR, OUT_DIR, MAP_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# Study area polygon (WGS84)
study_box_geo = box(BBOX_WEST, BBOX_SOUTH, BBOX_EAST, BBOX_NORTH)
study_gdf_geo = gpd.GeoDataFrame({'geometry': [study_box_geo]}, crs=CRS_GEO)

print("Project configuration:")
print(f"  Bounding box : {BBOX_WEST}°E – {BBOX_EAST}°E,  {BBOX_SOUTH}°N – {BBOX_NORTH}°N")
print(f"  Area (approx): ~{(BBOX_EAST - BBOX_WEST)*111:.0f} × {(BBOX_NORTH - BBOX_SOUTH)*111:.0f} km")
print(f"  Analysis CRS : {CRS_PROJ}")
print(f"  Grid cell    : {RESOLUTION_M} m")
print(f"  Base dir     : {BASE_DIR}")


## Stage 3: DATA ACQUISITION


In [ ]:
print("\n" + "="*70)
print("STAGE 3 — DATA ACQUISITION")
print("="*70)

# ── Download utilities ──────────────────────────────────────────────────────
def download_file(url, dest_path, description='file', chunk_size=1024*1024):
    """Download a file with resume support, integrity validation and progress display."""
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        print(f"  [SKIP] {description} already downloaded ({os.path.getsize(dest_path)/1e6:.1f} MB).")
        return dest_path
    print(f"  Downloading {description}...")
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    temp_path = dest_path + ".part"
    try:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            downloaded = 0
            with open(temp_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            pct = downloaded / total * 100
                            print(f"\r    {downloaded/1e6:.1f}/{total/1e6:.1f} MB ({pct:.0f}%)",
                                  end='', flush=True)
            if total and downloaded < total:
                raise IOError(f"Incomplete download: {downloaded}/{total} bytes received.")
        if os.path.exists(dest_path):
            os.remove(dest_path)
        os.rename(temp_path, dest_path)
        print(f"\n  [OK] Saved: {dest_path}")
        return dest_path
    except Exception as e:
        print(f"\n  [ERROR] Failed to download {description}: {e}")
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise

def extract_zip(zip_path, extract_dir):
    """Extract a ZIP archive."""
    os.makedirs(extract_dir, exist_ok=True)
    print(f"  Extracting {os.path.basename(zip_path)}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_dir)
    print(f"  Extracted to: {extract_dir}")


# ── 3A. SRTM 30m DEM via OpenTopography ────────────────────────────────────
# Provider  : NASA / USGS via OpenTopography
# API docs  : https://portal.opentopography.org/apidocs/
# Auth      : Optional — register free at https://opentopography.org/
# License   : NASA Open Data Policy

OT_API_KEY = ''   # OPTIONAL: paste your free API key here
srtm_path = os.path.join(DATA_DIR, 'srtm_lokoja.tif')

if not os.path.exists(srtm_path):
    ot_url = (
        f"https://portal.opentopography.org/API/globaldem"
        f"?demtype=SRTMGL1"
        f"&west={BBOX_WEST}&east={BBOX_EAST}"
        f"&south={BBOX_SOUTH}&north={BBOX_NORTH}"
        f"&outputFormat=GTiff"
    )
    if OT_API_KEY:
        ot_url += f"&API_Key={OT_API_KEY}"

    print("Downloading SRTM 30m DEM from OpenTopography...")
    try:
        resp = requests.get(ot_url, timeout=300)
        content_type = resp.headers.get('Content-Type', '')
        if resp.status_code == 200 and ('tiff' in content_type or 'octet' in content_type):
            with open(srtm_path, 'wb') as f:
                f.write(resp.content)
            print(f"  [OK] SRTM saved: {srtm_path} ({os.path.getsize(srtm_path)/1e6:.1f} MB)")
        else:
            print(f"  [WARN] OpenTopography response: {resp.status_code}")
            print(f"  Body: {resp.text[:400]}")
            raise ValueError("OpenTopography download failed")
    except Exception as e:
        print(f"  OpenTopography failed: {e}")
        print()
        print("  ─── MANUAL SRTM DOWNLOAD INSTRUCTIONS ───")
        print("  Option 1: USGS EarthExplorer (recommended)")
        print("    URL  : https://earthexplorer.usgs.gov/")
        print("    Coords: N 8.20, S 7.40, W 6.40, E 7.20")
        print("    Dataset: SRTM > SRTM 1 Arc-Second Global")
        print(f"    Save to: {srtm_path}")
        print()
        print("  Option 2: CGIAR-CSI SRTM 90m (no auth required)")
        # Tile 38_11 covers 5-10°E, 5-10°N — includes Lokoja/Kogi State, Nigeria
        cgiar_url = 'https://srtm.csi.cgiar.org/wp-content/uploads/files/srtm_5x5/TIFF/srtm_38_11.zip'
        cgiar_zip = os.path.join(DATA_DIR, 'srtm_cgiar.zip')
        try:
            download_file(cgiar_url, cgiar_zip, 'SRTM 90m (CGIAR tile 38_11 — Nigeria)')
            extract_zip(cgiar_zip, DATA_DIR)
            for fn in os.listdir(DATA_DIR):
                if fn.endswith('.tif') and 'srtm' in fn.lower() and fn != 'srtm_lokoja.tif':
                    os.rename(os.path.join(DATA_DIR, fn), srtm_path)
                    print(f"  Renamed to: {srtm_path}")
                    break
        except Exception as e2:
            print(f"  CGIAR download also failed: {e2}")
            print(f"  Please download SRTM manually and save to: {srtm_path}")
else:
    print(f"[SKIP] SRTM already downloaded.")

if os.path.exists(srtm_path):
    with rasterio.open(srtm_path) as src:
        _d = src.read(1)
        _valid = _d[_d != src.nodata] if src.nodata else _d.ravel()
        print(f"SRTM DEM: CRS={src.crs}, Res={src.res}, "
              f"Shape={src.width}x{src.height}, "
              f"Elevation range: {_valid.min():.0f}–{_valid.max():.0f} m")


# ── 3B. CHIRPS Monthly Rainfall — September & October 2022 ─────────────────
# Provider  : Climate Hazards Center, UC Santa Barbara / USGS
# Source    : https://www.chc.ucsb.edu/data/chirps
# License   : CC0 (public domain)
# Resolution: 0.05 degrees (~5.3 km)

CHIRPS_BASE = 'https://data.chc.ucsb.edu/products/CHIRPS-2.0/africa_monthly/tifs'
CHIRPS_MONTHS = [('2022', '09'), ('2022', '10')]

chirps_files = []
for year, month in CHIRPS_MONTHS:
    remote = f"chirps-v2.0.{year}.{month}.tif.gz"
    gz_path  = os.path.join(DATA_DIR, f"chirps_{year}{month}.tif.gz")
    tif_path = os.path.join(DATA_DIR, f"chirps_{year}{month}.tif")
    if not os.path.exists(tif_path):
        download_file(f"{CHIRPS_BASE}/{remote}", gz_path, f"CHIRPS {year}-{month}")
        print(f"  Decompressing...")
        with gzip.open(gz_path, 'rb') as f_in, open(tif_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path)
        print(f"  [OK] {tif_path}")
    else:
        print(f"[SKIP] CHIRPS {year}-{month} already downloaded.")
    chirps_files.append(tif_path)


# ── 3C. HydroRIVERS Africa ──────────────────────────────────────────────────
# Provider  : HydroSHEDS / WWF
# Source    : https://www.hydrosheds.org/products/hydrorivers
# License   : CC BY 4.0

hydro_dir = os.path.join(DATA_DIR, 'hydrorivers')
hydro_zip = os.path.join(DATA_DIR, 'HydroRIVERS_v10_af.zip')
hydro_url = 'https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_af_shp.zip'

if not os.path.exists(hydro_dir):
    download_file(hydro_url, hydro_zip, 'HydroRIVERS Africa')
    extract_zip(hydro_zip, hydro_dir)
else:
    print("[SKIP] HydroRIVERS already extracted.")

hydro_shp = None
for root, _, files in os.walk(hydro_dir):
    for f in files:
        if f.endswith('.shp') and 'HydroRIVERS' in f:
            hydro_shp = os.path.join(root, f)
            break

rivers_clip_path = os.path.join(DATA_DIR, 'rivers_lokoja.gpkg')
if hydro_shp and not os.path.exists(rivers_clip_path):
    print("Clipping HydroRIVERS to study area...")
    rivers_raw = gpd.read_file(hydro_shp, bbox=(BBOX_WEST, BBOX_SOUTH, BBOX_EAST, BBOX_NORTH))
    rivers_raw.to_file(rivers_clip_path, driver='GPKG')
    print(f"  River reaches in study area: {len(rivers_raw)}")
elif os.path.exists(rivers_clip_path):
    print("[SKIP] Clipped rivers already saved.")


# ── 3D. ESA WorldCover 2021 (10m Land Cover) ───────────────────────────────
# Provider  : ESA / Terrascope (public S3 bucket)
# Source    : https://esa-worldcover.org/en
# License   : CC BY 4.0
# Tile N06E006 covers Lokoja area (6-9N, 6-9E)

WC_TILE = 'N06E006'
WC_URL  = (
    f"https://esa-worldcover.s3.amazonaws.com/v200/2021/map/"
    f"ESA_WorldCover_10m_2021_v200_{WC_TILE}_Map.tif"
)
wc_raw_path = os.path.join(DATA_DIR, f"worldcover_{WC_TILE}.tif")

if not os.path.exists(wc_raw_path):
    download_file(WC_URL, wc_raw_path, f"ESA WorldCover tile {WC_TILE}")
else:
    print("[SKIP] ESA WorldCover already downloaded.")

# WorldCover class lookup
WC_CLASSES = {
    10: 'Tree cover',        20: 'Shrubland',
    30: 'Grassland',         40: 'Cropland',
    50: 'Built-up',          60: 'Bare/sparse vegetation',
    70: 'Snow/ice',          80: 'Permanent water bodies',
    90: 'Herbaceous wetland', 95: 'Mangroves',
    100: 'Moss/lichen'
}


# ── 3E. UNOSAT Flood Extent — Nigeria October 2022 ─────────────────────────
# Provider  : UNITAR-UNOSAT (Product ID: 3427)
# Activation: FL20221019NGA (Initial emergency alert: FL20221014NGA)
# Sensor    : VIIRS (Visible Infrared Imaging Radiometer Suite on Suomi NPP / NOAA-20)
# Product   : Maximum Flood Water Extent composite (2022-10-01 to 2022-10-25)
# Spatial Res: 375m native resolution, vectorized by UNOSAT GIS analysts
# Source URL: https://unosat.org/static/unosat_filesystem/3427/FL20221019NGA_SHP.zip
# License   : Creative Commons Attribution (CC BY-IGO 3.0)

unosat_zip = os.path.join(DATA_DIR, 'unosat_flood_nga_2022.zip')
unosat_dir = os.path.join(DATA_DIR, 'unosat')
UNOSAT_URL = "https://unosat.org/static/unosat_filesystem/3427/FL20221019NGA_SHP.zip"

flood_clip_path = os.path.join(DATA_DIR, 'flood_extent_lokoja.gpkg')

if not os.path.exists(unosat_dir):
    try:
        download_file(UNOSAT_URL, unosat_zip, 'UNOSAT Flood Extent (Kogi)')
        extract_zip(unosat_zip, unosat_dir)
    except Exception as e:
        print(f"  [WARN] Direct download failed: {e}")
        print()
        print("  ─── MANUAL UNOSAT DOWNLOAD INSTRUCTIONS ───")
        print("  1. Visit: https://data.humdata.org/organization/unosat")
        print("  2. Search: FL20221014NGA Nigeria flood 2022 Kogi")
        print("  3. Download the shapefile ZIP")
        print(f"  4. Extract to: {unosat_dir}")
        print()
        print("  Alternative: https://unosat.org/products/")
else:
    print("[SKIP] UNOSAT data already extracted.")

# Select MaximumFloodWaterExtent for peak October 2022 flood period
# Priority order: Oct 1-25 (broadest), then Oct 13-17, then Oct 19-23
FLOOD_SHP_PRIORITY = [
    'VIIRS_20221001_20221025_MaximumFloodWaterExtent_Nigeria.shp',
    'VIIRS_20221013_20221017_MaximumFloodWaterExtent_Nigeria.shp',
    'VIIRS_20221019_20221023_MaximumFloodWaterExtent_Nigeria.shp',
]
unosat_shp = None
if os.path.exists(unosat_dir):
    for root, _, files in os.walk(unosat_dir):
        for preferred in FLOOD_SHP_PRIORITY:
            if preferred in files:
                unosat_shp = os.path.join(root, preferred)
                print(f"  Using UNOSAT shapefile: {preferred}")
                break
        if unosat_shp:
            break
    # Fallback: any MaximumFloodWaterExtent shapefile
    if not unosat_shp:
        for root, _, files in os.walk(unosat_dir):
            for f in files:
                if 'MaximumFloodWaterExtent' in f and f.endswith('.shp'):
                    unosat_shp = os.path.join(root, f)
                    print(f"  Using UNOSAT shapefile (fallback): {f}")
                    break
            if unosat_shp:
                break

if unosat_shp and not os.path.exists(flood_clip_path):
    flood_gdf = gpd.read_file(unosat_shp).to_crs(CRS_GEO)
    flood_gdf = gpd.clip(flood_gdf, study_gdf_geo)
    flood_gdf.to_file(flood_clip_path, driver='GPKG')
    print(f"UNOSAT flood extent clipped: {len(flood_gdf)} polygons in study area")
elif os.path.exists(flood_clip_path):
    print("[SKIP] Flood extent already clipped.")
else:
    print("[INFO] UNOSAT not yet available — complete manual download above.")


## Stage 4: DATASET UNDERSTANDING


In [ ]:
print("\n" + "="*70)
print("STAGE 4 — DATASET UNDERSTANDING")
print("="*70)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Stage 4: Raw Dataset Overview — Lokoja Study Area", fontsize=14, fontweight='bold')

# 4A — Elevation
if os.path.exists(srtm_path):
    with rasterio.open(srtm_path) as src:
        dem_raw = src.read(1).astype(float)
        if src.nodata: dem_raw[dem_raw == src.nodata] = np.nan
    im = axes[0].imshow(dem_raw, cmap='terrain', aspect='auto')
    plt.colorbar(im, ax=axes[0], label='Elevation (m)')
    axes[0].set_title("SRTM Elevation (30m)", fontweight='bold')

# 4B — CHIRPS Rainfall
if chirps_files and os.path.exists(chirps_files[0]):
    with rasterio.open(chirps_files[0]) as src:
        from rasterio.windows import from_bounds as wfb
        win = wfb(BBOX_WEST, BBOX_SOUTH, BBOX_EAST, BBOX_NORTH, src.transform)
        cr = src.read(1, window=win).astype(float)
        cr[cr < 0] = np.nan
    im = axes[1].imshow(cr, cmap='Blues', aspect='auto')
    plt.colorbar(im, ax=axes[1], label='Rainfall (mm)')
    axes[1].set_title("CHIRPS Sep-2022 (~5.3km)", fontweight='bold')

# 4C — WorldCover
if os.path.exists(wc_raw_path):
    with rasterio.open(wc_raw_path) as src:
        from rasterio.windows import from_bounds as wfb
        win = wfb(BBOX_WEST, BBOX_SOUTH, BBOX_EAST, BBOX_NORTH, src.transform)
        wc = src.read(1, window=win)
    im = axes[2].imshow(wc, cmap='tab20', aspect='auto', vmin=10, vmax=100)
    plt.colorbar(im, ax=axes[2], label='Land Cover Class')
    axes[2].set_title("ESA WorldCover 2021 (10m)", fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'stage4_raw_datasets.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: stage4_raw_datasets.png")


## Stage 5: SPATIAL PREPROCESSING


In [ ]:
print("\n" + "="*70)
print("STAGE 5 — SPATIAL PREPROCESSING")
print("="*70)
print("All datasets reprojected to UTM Zone 32N (EPSG:32632), 90m resolution.")
print("Rationale: Metric CRS required for slope, distance, and area calculations.")

# ── Build 90m UTM master grid ───────────────────────────────────────────────
t = Transformer.from_crs(CRS_GEO, CRS_PROJ, always_xy=True)
x_min, y_min = t.transform(BBOX_WEST, BBOX_SOUTH)
x_max, y_max = t.transform(BBOX_EAST, BBOX_NORTH)

x_min = int(x_min // RESOLUTION_M) * RESOLUTION_M
y_min = int(y_min // RESOLUTION_M) * RESOLUTION_M
x_max = int(x_max // RESOLUTION_M + 1) * RESOLUTION_M
y_max = int(y_max // RESOLUTION_M + 1) * RESOLUTION_M

n_cols = int((x_max - x_min) / RESOLUTION_M)
n_rows = int((y_max - y_min) / RESOLUTION_M)

MASTER_TRANSFORM = rasterio.transform.from_origin(x_min, y_max, RESOLUTION_M, RESOLUTION_M)
MASTER_CRS       = CRS_PROJ
MASTER_SHAPE     = (n_rows, n_cols)

print(f"\nMaster grid: {n_cols}c × {n_rows}r = {n_cols*n_rows:,} pixels")
print(f"Extent: {(x_max-x_min)/1000:.1f} × {(y_max-y_min)/1000:.1f} km")


# ── Reprojection utility ────────────────────────────────────────────────────
def reproject_to_master(src_path, out_path,
                         resampling=Resampling.bilinear, nodata=-9999.0):
    """Reproject any raster to the master UTM 90m grid."""
    if os.path.exists(out_path):
        return out_path
    with rasterio.open(src_path) as src:
        meta = src.meta.copy()
        meta.update({
            'crs': MASTER_CRS, 'transform': MASTER_TRANSFORM,
            'width': MASTER_SHAPE[1], 'height': MASTER_SHAPE[0],
            'nodata': nodata, 'dtype': 'float32', 'count': 1,
        })
        with rasterio.open(out_path, 'w', **meta) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=MASTER_TRANSFORM, dst_crs=MASTER_CRS,
                resampling=resampling,
            )
    print(f"  [OK] Reprojected: {os.path.basename(out_path)}")
    return out_path


# ── 5A. SRTM Elevation ─────────────────────────────────────────────────────
dem_proj_path = os.path.join(DATA_DIR, 'dem_90m_utm32n.tif')
if os.path.exists(srtm_path):
    reproject_to_master(srtm_path, dem_proj_path)
else:
    print("[WARN] SRTM not available — complete Stage 3A first.")


# ── 5B. CHIRPS Rainfall accumulation (Sep + Oct 2022) ──────────────────────
chirps_proj = []
for i, (year, month) in enumerate(CHIRPS_MONTHS):
    src_p = chirps_files[i]
    out_p = os.path.join(DATA_DIR, f"chirps_{year}{month}_90m.tif")
    if os.path.exists(src_p):
        reproject_to_master(src_p, out_p)
    chirps_proj.append(out_p)

rain_accum_path = os.path.join(DATA_DIR, 'rain_sepoct2022_90m.tif')
if all(os.path.exists(p) for p in chirps_proj) and not os.path.exists(rain_accum_path):
    arrs, meta = [], None
    for p in chirps_proj:
        with rasterio.open(p) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr < 0] = np.nan
            arrs.append(arr)
            if meta is None: meta = src.meta.copy()
    accum = np.nansum(arrs, axis=0).astype(np.float32)
    accum[np.all(np.isnan(arrs), axis=0)] = np.nan
    meta.update({'dtype': 'float32', 'nodata': -9999})
    out = accum.copy(); out[np.isnan(accum)] = -9999
    with rasterio.open(rain_accum_path, 'w', **meta) as dst:
        dst.write(out, 1)
    print(f"Sep+Oct accumulation: {np.nanmin(accum):.0f}–{np.nanmax(accum):.0f} mm, "
          f"mean={np.nanmean(accum):.0f} mm")


# ── 5C. ESA WorldCover (nearest-neighbour for categorical data) ─────────────
wc_proj_path = os.path.join(DATA_DIR, 'worldcover_90m_utm32n.tif')
if os.path.exists(wc_raw_path):
    reproject_to_master(wc_raw_path, wc_proj_path, Resampling.nearest, nodata=0)


# ── 5D. Rasterize UNOSAT Flood Extent ──────────────────────────────────────
flood_raster_path = os.path.join(DATA_DIR, 'flood_mask_90m_utm32n.tif')

if os.path.exists(flood_clip_path) and not os.path.exists(flood_raster_path):
    fd = gpd.read_file(flood_clip_path).to_crs(MASTER_CRS)
    shapes = [(geom, 1) for geom in fd.geometry if geom is not None]
    flood_raster = rasterize(
        shapes=shapes, out_shape=MASTER_SHAPE,
        transform=MASTER_TRANSFORM, fill=0, dtype=np.uint8, all_touched=False
    )
    meta = {
        'driver': 'GTiff', 'dtype': 'uint8',
        'crs': MASTER_CRS, 'transform': MASTER_TRANSFORM,
        'width': MASTER_SHAPE[1], 'height': MASTER_SHAPE[0],
        'count': 1, 'nodata': 255,
    }
    with rasterio.open(flood_raster_path, 'w', **meta) as dst:
        dst.write(flood_raster, 1)
    print(f"Flood rasterized: {np.sum(flood_raster==1):,} flooded, "
          f"{np.sum(flood_raster==0):,} dry")
elif os.path.exists(flood_raster_path):
    print("[SKIP] Flood raster already exists.")


## Stage 6: DATA INTEGRATION


In [ ]:
print("\n" + "="*70)
print("STAGE 6 — DATA INTEGRATION")
print("="*70)

def load_raster(path, nodata=None):
    """Load raster band 1 as float32 with NaN for nodata."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nd  = nodata if nodata is not None else src.nodata
        if nd is not None:
            arr[arr == nd] = np.nan
    return arr

raster_map = {
    'elevation': dem_proj_path,
    'rainfall' : rain_accum_path,
    'lulc'     : wc_proj_path,
}

feature_arrays = {}
for name, path in raster_map.items():
    if os.path.exists(path):
        feature_arrays[name] = load_raster(path)
        arr = feature_arrays[name]
        pct = np.sum(~np.isnan(arr)) / arr.size * 100
        print(f"  {name:<15} shape={arr.shape}  "
              f"valid={pct:.1f}%  "
              f"range=[{np.nanmin(arr):.1f}, {np.nanmax(arr):.1f}]")
    else:
        print(f"  [MISSING] {name}: {path}")


## Stage 7: DATA CLEANING


In [ ]:
print("\n" + "="*70)
print("STAGE 7 — DATA CLEANING")
print("="*70)

cleaning_log = []
for name, arr in feature_arrays.items():
    n_total = arr.size
    n_nan   = int(np.sum(np.isnan(arr)))
    pct_nan = n_nan / n_total * 100

    print(f"\n{name.upper()}")
    print(f"  Total cells  : {n_total:,}")
    print(f"  NaN count    : {n_nan:,} ({pct_nan:.2f}%)")
    if not np.isnan(arr).all():
        print(f"  Min / Max    : {np.nanmin(arr):.2f} / {np.nanmax(arr):.2f}")
        print(f"  Mean / Std   : {np.nanmean(arr):.2f} / {np.nanstd(arr):.2f}")

    if name == 'lulc':
        bad = arr[(arr > 0) & (~np.isin(arr, list(WC_CLASSES.keys())))]
        if len(bad) > 0:
            print(f"  [WARN] Unexpected LULC values: {np.unique(bad)}")
        else:
            print(f"  [OK] All LULC classes are valid WorldCover values.")

    if name == 'elevation':
        n_neg = int(np.sum(arr < 0))
        if n_neg > 0:
            print(f"  [WARN] {n_neg} cells with negative elevation — DEM boundary artefact.")
            print(f"  Decision: retain; physically unlikely at inland Nigerian locations.")
        else:
            print(f"  [OK] No negative elevations.")

    cleaning_log.append({
        'Feature'    : name,
        'Total cells': n_total,
        'NaN count'  : n_nan,
        'NaN %'      : round(pct_nan, 2),
        'Min'        : round(float(np.nanmin(arr)), 2) if not np.isnan(arr).all() else None,
        'Max'        : round(float(np.nanmax(arr)), 2) if not np.isnan(arr).all() else None,
    })

# ── Cleaning decisions ─────────────────────────────────────────────────────
# DECISION: Mask permanent water bodies (WorldCover class 80).
# Reason: Pre-existing water channels are not flood expansion events.
#         Including them as y=1 would teach the model "where the river is",
#         not "where flooding expanded to".

if 'lulc' in feature_arrays:
    water_mask = (feature_arrays['lulc'] == 80)
    print(f"\nPermanent water cells masked: {np.sum(water_mask):,}")
else:
    water_mask = np.zeros(MASTER_SHAPE, dtype=bool)

# DECISION: Exclude any pixel with NaN in ANY feature layer.
# Reason: A sample cannot be presented to the model with missing values
#         without imputation. Imputation at pixel level is risky for spatial data.
valid_mask = ~water_mask
for arr in feature_arrays.values():
    valid_mask &= ~np.isnan(arr)

n_valid = int(np.sum(valid_mask))
print(f"Valid modelling pixels: {n_valid:,} ({n_valid/valid_mask.size*100:.1f}% of grid)")

print("\nCleaning Log:")
print(pd.DataFrame(cleaning_log).to_string(index=False))


## Stage 8: EXPLORATORY DATA ANALYSIS


In [ ]:
print("\n" + "="*70)
print("STAGE 8 — EXPLORATORY DATA ANALYSIS")
print("="*70)

# ── 8A. Spatial overview maps ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Stage 8: Spatial EDA — Lokoja Study Area\n"
             "Kogi State, Nigeria | Reference: October 2022 Flood",
             fontsize=14, fontweight='bold')

panels = {
    (0,0): ('elevation', 'terrain',   'Elevation (m)',       'A. Elevation (SRTM 90m)'),
    (0,1): ('rainfall',  'Blues',     'Rainfall mm',         'B. Rainfall Sep–Oct 2022 (CHIRPS)'),
    (1,0): ('lulc',      'tab20',     'WorldCover Class',    'C. Land Cover (ESA WorldCover 2021)'),
}

for (r,c), (feat, cmap, clabel, title) in panels.items():
    ax = axes[r][c]
    if feat in feature_arrays:
        im = ax.imshow(feature_arrays[feat], cmap=cmap, aspect='auto',
                       vmin=(10 if feat=='lulc' else None),
                       vmax=(100 if feat=='lulc' else None))
        plt.colorbar(im, ax=ax, label=clabel)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("West → East")
    if c == 0: ax.set_ylabel("South → North")

# Flood mask
ax = axes[1][1]
if os.path.exists(flood_raster_path):
    with rasterio.open(flood_raster_path) as src:
        fd = src.read(1).astype(float)
        fd[fd == 255] = np.nan
    cmap_f = mcolors.ListedColormap(['#D4E6F1', '#E74C3C'])
    im = ax.imshow(fd, cmap=cmap_f, aspect='auto', vmin=0, vmax=1)
    patches = [mpatches.Patch(color='#D4E6F1', label='Dry (y=0)'),
               mpatches.Patch(color='#E74C3C', label='Flooded (y=1)')]
    ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_title("D. UNOSAT Flood Extent — Oct 2022 (Labels)", fontweight='bold')
ax.set_xlabel("West → East")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'stage8_spatial_eda.png'), dpi=150, bbox_inches='tight')
plt.show()


# ── 8B. Build EDA DataFrame ────────────────────────────────────────────────
rows_idx, cols_idx = np.where(valid_mask)
eda_data = {'row': rows_idx, 'col': cols_idx}
for name in feature_arrays:
    eda_data[name] = feature_arrays[name][rows_idx, cols_idx]

if os.path.exists(flood_raster_path):
    with rasterio.open(flood_raster_path) as src:
        flood_arr_full = src.read(1)
    eda_data['flood'] = flood_arr_full[rows_idx, cols_idx].astype(int)

df_eda = pd.DataFrame(eda_data)
print(f"\nEDA DataFrame: {len(df_eda):,} valid pixels")
print(df_eda.describe().round(2))

if 'flood' in df_eda.columns:
    vc = df_eda['flood'].value_counts().sort_index()
    print(f"\nTarget class distribution:")
    for cls, cnt in vc.items():
        lbl = 'Flooded' if cls == 1 else 'Dry    '
        print(f"  y={cls} ({lbl}): {cnt:>8,}  ({cnt/len(df_eda)*100:.1f}%)")
    print(f"  Imbalance: {vc.get(0,0)/max(vc.get(1,1),1):.0f}:1 (dry:flooded)")


# ── 8C. Feature distributions by flood class ──────────────────────────────
if 'flood' in df_eda.columns:
    num_feats = [f for f in ['elevation', 'rainfall'] if f in df_eda.columns]
    fig, axes2 = plt.subplots(1, len(num_feats), figsize=(7*len(num_feats), 5))
    if len(num_feats) == 1: axes2 = [axes2]
    fig.suptitle("Feature Distributions: Flooded vs Dry", fontsize=13, fontweight='bold')

    for ax, feat in zip(axes2, num_feats):
        dry_s = df_eda[df_eda['flood']==0][feat].dropna()
        fld_s = df_eda[df_eda['flood']==1][feat].dropna()
        if len(dry_s) > 20000: dry_s = dry_s.sample(20000, random_state=RANDOM_STATE)
        if len(fld_s) > 20000: fld_s = fld_s.sample(20000, random_state=RANDOM_STATE)

        ax.hist(dry_s, bins=50, alpha=0.65, color='#3498DB', label='Dry (y=0)', density=True)
        ax.hist(fld_s, bins=50, alpha=0.65, color='#E74C3C', label='Flooded (y=1)', density=True)
        ax.set_xlabel(feat.replace('_', ' ').title())
        ax.set_ylabel('Density')
        ax.set_title(f"{feat.replace('_',' ').title()} Distribution")
        ax.legend(); ax.grid(alpha=0.3)

        print(f"\n{feat}: Dry median={dry_s.median():.1f}, Flooded median={fld_s.median():.1f}")

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'stage8_distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()


# ── 8D. Correlation matrix ─────────────────────────────────────────────────
num_cols = [c for c in df_eda.columns if c not in ['row','col','lulc']]
if len(num_cols) > 1:
    samp = df_eda[num_cols].dropna()
    if len(samp) > 50000: samp = samp.sample(50000, random_state=RANDOM_STATE)
    corr = samp.corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title("Correlation Matrix — Flood Features", fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'stage8_correlation.png'), dpi=150, bbox_inches='tight')
    plt.show()


## Stage 9: FEATURE ENGINEERING


In [ ]:
print("\n" + "="*70)
print("STAGE 9 — FEATURE ENGINEERING")
print("="*70)
print("Deriving 3 additional features from raw datasets.")

# ── 9A. Slope (degrees) from SRTM DEM ─────────────────────────────────────
# Feature  : Slope
# Source   : SRTM DEM (dem_90m_utm32n.tif)
# Method   : Central finite-difference gradient
# Purpose  : Flat terrain retains water; steep slopes drain rapidly
# Limitation: 90m DEM smooths fine-scale channel features

slope_path = os.path.join(DATA_DIR, 'slope_90m_utm32n.tif')

if os.path.exists(dem_proj_path) and not os.path.exists(slope_path):
    dem_s = load_raster(dem_proj_path)
    dem_f = dem_s.copy()
    dem_f[np.isnan(dem_f)] = np.nanmean(dem_f)   # fill NaN for gradient

    dz_dy, dz_dx = np.gradient(dem_f, RESOLUTION_M, RESOLUTION_M)
    slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))).astype(np.float32)
    slope_deg[np.isnan(dem_s)] = np.nan

    with rasterio.open(dem_proj_path) as ref:
        meta = ref.meta.copy()
        meta.update({'dtype': 'float32', 'nodata': -9999})
    slope_out = slope_deg.copy(); slope_out[np.isnan(slope_deg)] = -9999
    with rasterio.open(slope_path, 'w', **meta) as dst:
        dst.write(slope_out, 1)
    print(f"Slope: {np.nanmin(slope_deg):.2f}°–{np.nanmax(slope_deg):.2f}°, "
          f"mean={np.nanmean(slope_deg):.2f}°")
    feature_arrays['slope'] = slope_deg
elif os.path.exists(slope_path):
    feature_arrays['slope'] = load_raster(slope_path, nodata=-9999)
    print("[SKIP] Slope already computed.")
else:
    print("[WARN] DEM not available for slope computation.")


# ── 9B. Distance to Nearest River (metres) ────────────────────────────────
# Feature  : dist_river
# Source   : HydroRIVERS (rivers_lokoja.gpkg)
# Method   : Euclidean distance transform on rasterized river lines
# Purpose  : Proximity to drainage channels is a primary flood predictor
# Limitation: Only horizontal distance; does not capture river discharge capacity

dist_river_path = os.path.join(DATA_DIR, 'dist_river_90m_utm32n.tif')

if not os.path.exists(dist_river_path):
    if os.path.exists(rivers_clip_path):
        riv_proj = gpd.read_file(rivers_clip_path).to_crs(MASTER_CRS)
        shapes = [(geom, 1) for geom in riv_proj.geometry if geom is not None]
        riv_raster = rasterize(
            shapes=shapes, out_shape=MASTER_SHAPE,
            transform=MASTER_TRANSFORM, fill=0, dtype=np.uint8, all_touched=True
        )
        dist_pix = distance_transform_edt(riv_raster == 0)
        dist_m   = (dist_pix * RESOLUTION_M).astype(np.float32)

        with rasterio.open(dem_proj_path) as ref:
            meta = ref.meta.copy()
            meta.update({'dtype': 'float32', 'nodata': -9999})
        with rasterio.open(dist_river_path, 'w', **meta) as dst:
            dst.write(dist_m, 1)
        print(f"Distance to river: {dist_m.min():.0f}–{dist_m.max():.0f} m, "
              f"mean={dist_m.mean():.0f} m")
        feature_arrays['dist_river'] = dist_m
    else:
        print("[WARN] HydroRIVERS not available for distance computation.")
else:
    feature_arrays['dist_river'] = load_raster(dist_river_path, nodata=-9999)
    print("[SKIP] Distance-to-river already computed.")


# ── 9C. Permeability Encoding from WorldCover ──────────────────────────────
# Feature  : permeability
# Source   : ESA WorldCover 2021 (worldcover_90m_utm32n.tif)
# Method   : Ordinal encoding (0=impermeable, 10=highly permeable)
# Purpose  : Surface impermeability increases runoff → higher flood risk
# Limitation: Approximate; actual infiltration varies within class

PERM_MAP = {
    50:1, 80:2, 60:3, 40:5, 90:5, 95:6, 30:6, 20:7, 10:8, 70:9, 100:9
}

perm_path = os.path.join(DATA_DIR, 'permeability_90m_utm32n.tif')

if not os.path.exists(perm_path) and 'lulc' in feature_arrays:
    lulc = feature_arrays['lulc'].copy()
    perm = np.full_like(lulc, np.nan)
    for cls, val in PERM_MAP.items():
        perm[lulc == cls] = val
    unknown = (~np.isnan(lulc)) & np.isnan(perm) & (lulc > 0)
    if unknown.any():
        perm[unknown] = 5  # neutral default for unknown classes
    with rasterio.open(wc_proj_path) as ref:
        meta = ref.meta.copy()
        meta.update({'dtype': 'float32', 'nodata': -9999})
    po = perm.copy(); po[np.isnan(perm)] = -9999
    with rasterio.open(perm_path, 'w', **meta) as dst:
        dst.write(po, 1)
    print("Permeability encoded and saved.")
    feature_arrays['permeability'] = perm
elif os.path.exists(perm_path):
    feature_arrays['permeability'] = load_raster(perm_path, nodata=-9999)
    print("[SKIP] Permeability already encoded.")

print("\nFinal feature set:")
for n, a in feature_arrays.items():
    if n != 'lulc':
        print(f"  {n}")


## Stage 10: TARGET CONSTRUCTION


In [ ]:
print("\n" + "="*70)
print("STAGE 10 — TARGET CONSTRUCTION")
print("="*70)

FEATURE_NAMES = [f for f in
    ['elevation', 'slope', 'rainfall', 'dist_river', 'permeability']
    if f in feature_arrays]

print(f"Modelling features: {FEATURE_NAMES}")

# Rebuild valid mask with all engineered features
valid_mask_final = ~water_mask.copy()
for name in FEATURE_NAMES:
    valid_mask_final &= ~np.isnan(feature_arrays[name])

if os.path.exists(flood_raster_path):
    with rasterio.open(flood_raster_path) as src:
        flood_arr_full = src.read(1)
    valid_mask_final &= (flood_arr_full != 255)

rows_m, cols_m = np.where(valid_mask_final)
model_data = {'row': rows_m, 'col': cols_m}
for name in FEATURE_NAMES:
    model_data[name] = feature_arrays[name][rows_m, cols_m]

if os.path.exists(flood_raster_path):
    model_data['flood'] = flood_arr_full[rows_m, cols_m].astype(int)

df_model = pd.DataFrame(model_data)
print(f"Modelling DataFrame: {len(df_model):,} samples × {len(FEATURE_NAMES)} features")

if 'flood' in df_model.columns:
    vc = df_model['flood'].value_counts().sort_index()
    for cls, cnt in vc.items():
        lbl = 'Flooded' if cls == 1 else 'Dry    '
        print(f"  y={cls} ({lbl}): {cnt:>8,}  ({cnt/len(df_model)*100:.1f}%)")
    print(f"  Imbalance: {vc.get(0,0)/max(vc.get(1,1),1):.0f}:1")


## Stage 11: TRAIN / TEST PREPARATION (SPATIAL BLOCK SPLIT)


In [ ]:
print("\n" + "="*70)
print("STAGE 11 — SPATIAL BLOCK SPLIT")
print("="*70)
print("Using 2×2 geographic quadrant split to evaluate spatial generalisation.")
print("Test set = SE quadrant (geographically distinct holdout partition).")
print("Note: Quadrant split prevents direct train/test pixel overlap, though residual")
print("      spatial autocorrelation across immediate block boundaries may remain.")

mid_r = MASTER_SHAPE[0] // 2
mid_c = MASTER_SHAPE[1] // 2

def quadrant(r, c):
    return 0 if r<mid_r and c<mid_c else 1 if r<mid_r else 2 if c<mid_c else 3

df_model['quadrant'] = df_model.apply(
    lambda row: quadrant(row['row'], row['col']), axis=1)

print("\nSamples per quadrant:")
print(df_model['quadrant'].value_counts().sort_index().to_string())

TEST_QUAD = 3  # SE quadrant — documented design decision

df_train = df_model[df_model['quadrant'] != TEST_QUAD]
df_test  = df_model[df_model['quadrant'] == TEST_QUAD]

X_train = df_train[FEATURE_NAMES].values
X_test  = df_test[FEATURE_NAMES].values

if 'flood' in df_model.columns:
    y_train = df_train['flood'].values
    y_test  = df_test['flood'].values
    print(f"\nTrain: {len(df_train):,} ({len(df_train)/len(df_model)*100:.1f}%)")
    print(f"Test : {len(df_test):,}  ({len(df_test)/len(df_model)*100:.1f}%)")
    print(f"Train flood rate: {y_train.mean()*100:.2f}% (imbalance ~ {1/y_train.mean()-1:.1f}:1)")
    print(f"Test  flood rate: {y_test.mean()*100:.2f}% (imbalance ~ {1/y_test.mean()-1:.1f}:1)")
    print("\n[DOMAIN SHIFT AUDIT]")
    print(f"  Train elevation (mean ± std): {df_train['elevation'].mean():.1f} ± {df_train['elevation'].std():.1f} m (median {df_train['elevation'].median():.1f} m)")
    print(f"  Test  elevation (mean ± std): {df_test['elevation'].mean():.1f} ± {df_test['elevation'].std():.1f} m (median {df_test['elevation'].median():.1f} m)")
    print(f"  Train rainfall  (mean ± std): {df_train['rainfall'].mean():.1f} ± {df_train['rainfall'].std():.1f} mm")
    print(f"  Test  rainfall  (mean ± std): {df_test['rainfall'].mean():.1f} ± {df_test['rainfall'].std():.1f} mm")
    print(f"  Scientific note: Test set (SE quadrant) is an upland plateau region (+100.7m higher mean elevation),")
    print(f"  which physically drains water rapidly despite higher orographic rainfall, explaining why only")
    print(f"  1.57% of pixels were inundated. This represents a genuine spatial transferability / domain shift challenge.")
else:
    y_train = y_test = None
    print("[INFO] Labels not yet available — complete data acquisition first.")


## Stage 12: MODELLING


In [ ]:
print("\n" + "="*70)
print("STAGE 12 — MODELLING")
print("="*70)

if y_train is not None:
    # ── Baseline Model: Naive Majority-Class Classifier ──────────────────
    print("\nBaseline Model: Naive Majority-Class Classifier")
    print("  Reason: Mandatory benchmark for evaluating severely imbalanced holdout (98.4% dry).")
    dummy_model = DummyClassifier(strategy='most_frequent')
    dummy_model.fit(X_train, y_train)
    y_pred_base = dummy_model.predict(X_test)
    y_prob_base = np.zeros(len(y_test))  # positive class probability is 0
    print("  [OK] Majority-class baseline fitted.")

    # ── Model 1: Logistic Regression ──────────────────────────────────────
    print("\nModel 1: Logistic Regression (with StandardScaler)")
    print("  Reason: Interpretable linear baseline; "
          "coefficients show direction of feature effects.")
    lr_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(
            class_weight='balanced', max_iter=1000,
            random_state=RANDOM_STATE, solver='lbfgs'
        ))
    ])
    lr_pipeline.fit(X_train, y_train)
    y_pred_lr = lr_pipeline.predict(X_test)
    y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
    print("  [OK] Logistic Regression trained.")

    print("\n  Coefficients (positive -> higher flood probability):")
    coef = lr_pipeline.named_steps['lr'].coef_[0]
    for f, c in sorted(zip(FEATURE_NAMES, coef), key=lambda x: -x[1]):
        print(f"    {f:<20}: {c:+.4f}")

    print("\n  Multivariable Coefficient Diagnostic Analysis (Audit Finding Resolution):")
    print("    * elevation (-12.34): Dominant physical predictor; lowlands concentrate water.")
    print("    * slope (+1.66) & dist_river (+0.44): Positive multivariable coefficients result from")
    print("      Simpson's paradox / collinearity with elevation (r_elev,slope = +0.40, r_elev,dist = +0.43).")
    print("      In bivariate correlation, all features correlate negatively with flood (r_slope = -0.22,")
    print("      r_dist = -0.12). When elevation is fixed with massive negative weight (-12.34),")
    print("      the linear model uses slope and distance as suppressor terms for elevation over-fit.")
    print("    * rainfall (-0.58): Reflects regional fluvial vs pluvial hydrology. Flooding at Lokoja")
    print("      is driven by upstream river discharge (Niger + Benue) entering low-altitude valleys")
    print("      (mean 362mm local rain), whereas surrounding steep plateaus receive higher orographic")
    print("      rain (mean 468mm) but remain dry due to runoff. Random Forest avoids this linear artifact.")

    # ── Model 2: Random Forest ─────────────────────────────────────────────
    print("\nModel 2: Random Forest Classifier")
    print("  Reason: Captures non-linear relationships "
          "(e.g., elevation threshold below which flooding occurs).")
    rf_model = RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=50,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
    print("  [OK] Random Forest trained.")

    print("\n  Feature Importances (mean decrease in Gini impurity):")
    fi_pairs = sorted(zip(FEATURE_NAMES, rf_model.feature_importances_),
                      key=lambda x: -x[1])
    for f, imp in fi_pairs:
        print(f"    {f:<20}: {imp:.4f}")
else:
    print("[INFO] Labels not available — complete Stages 3–10 first.")


## Stage 13: EVALUATION


In [ ]:
print("\n" + "="*70)
print("STAGE 13 — EVALUATION")
print("="*70)

if y_train is not None:
    # Baseline metrics
    base_acc = accuracy_score(y_test, y_pred_base)
    base_roc_auc = 0.5000
    base_pr_auc = y_test.mean()  # Prevalence is horizontal line in PR space

    # Model metrics
    lr_roc_auc = roc_auc_score(y_test, y_prob_lr)
    lr_pr_auc  = average_precision_score(y_test, y_prob_lr)
    rf_roc_auc = roc_auc_score(y_test, y_prob_rf)
    rf_pr_auc  = average_precision_score(y_test, y_prob_rf)

    print("\n── Baseline (Majority Class: Predict Dry) ──")
    print(classification_report(y_test, y_pred_base,
          target_names=['Dry (0)', 'Flooded (1)'], digits=4, zero_division=0))
    print(f"ROC-AUC: {base_roc_auc:.4f} | PR-AUC (Baseline Prevalence): {base_pr_auc:.4f}")

    print("\n── Logistic Regression ──")
    print(classification_report(y_test, y_pred_lr,
          target_names=['Dry (0)', 'Flooded (1)'], digits=4))
    print(f"ROC-AUC: {lr_roc_auc:.4f} | PR-AUC: {lr_pr_auc:.4f} ({lr_pr_auc/base_pr_auc:.1f}x baseline)")

    print("\n── Random Forest ──")
    print(classification_report(y_test, y_pred_rf,
          target_names=['Dry (0)', 'Flooded (1)'], digits=4))
    print(f"ROC-AUC: {rf_roc_auc:.4f} | PR-AUC: {rf_pr_auc:.4f} ({rf_pr_auc/base_pr_auc:.1f}x baseline)")

    # ── Evaluation Plots (2x3 grid: CMs, ROC, PR, Feature Importance) ──
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle("Stage 13: Rigorous Model Evaluation — Baseline vs LR vs Random Forest",
                 fontsize=14, fontweight='bold')

    # CM Logistic Regression
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_lr),
        display_labels=['Dry','Flooded']).plot(ax=axes[0,0], colorbar=False)
    axes[0,0].set_title("Logistic Regression — Confusion Matrix", fontweight='bold')

    # CM Random Forest
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_rf),
        display_labels=['Dry','Flooded']).plot(ax=axes[0,1], colorbar=False)
    axes[0,1].set_title("Random Forest — Confusion Matrix", fontweight='bold')

    # CM Baseline
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_base),
        display_labels=['Dry','Flooded']).plot(ax=axes[0,2], colorbar=False)
    axes[0,2].set_title("Majority Baseline — Confusion Matrix", fontweight='bold')

    # ROC Curves
    ax_roc = axes[1,0]
    fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
    fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
    ax_roc.plot(fpr_lr, tpr_lr, color='#3498DB', lw=2,
                label=f"Logistic Reg (AUC = {lr_roc_auc:.3f})")
    ax_roc.plot(fpr_rf, tpr_rf, color='#E74C3C', lw=2,
                label=f"Random Forest (AUC = {rf_roc_auc:.3f})")
    ax_roc.plot([0,1], [0,1], 'k--', lw=1.5, label='Naive Baseline (AUC = 0.500)')
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("ROC Curves", fontweight='bold')
    ax_roc.legend(loc='lower right'); ax_roc.grid(alpha=0.3)

    # Precision-Recall Curves (Essential under extreme 98.4% imbalance)
    ax_pr = axes[1,1]
    p_lr, r_lr, _ = precision_recall_curve(y_test, y_prob_lr)
    p_rf, r_rf, _ = precision_recall_curve(y_test, y_prob_rf)
    ax_pr.plot(r_lr, p_lr, color='#3498DB', lw=2,
               label=f"Logistic Reg (PR-AUC = {lr_pr_auc:.3f})")
    ax_pr.plot(r_rf, p_rf, color='#E74C3C', lw=2,
               label=f"Random Forest (PR-AUC = {rf_pr_auc:.3f})")
    ax_pr.axhline(base_pr_auc, color='k', linestyle='--', lw=1.5,
                  label=f"Baseline Prevalence ({base_pr_auc:.3f})")
    ax_pr.set_xlabel("Recall (Flood Catch Rate)")
    ax_pr.set_ylabel("Precision (Positive Predictive Value)")
    ax_pr.set_title("Precision-Recall Curves", fontweight='bold')
    ax_pr.legend(loc='upper right'); ax_pr.grid(alpha=0.3)

    # Feature Importance
    ax_fi = axes[1,2]
    fi_sorted = sorted(zip(FEATURE_NAMES, rf_model.feature_importances_), key=lambda x: x[1])
    ax_fi.barh([f for f,_ in fi_sorted], [v for _,v in fi_sorted],
               color='#2ECC71', edgecolor='#27AE60')
    ax_fi.set_xlabel("Mean Decrease in Gini Impurity")
    ax_fi.set_title("Random Forest — Feature Importance", fontweight='bold')
    ax_fi.grid(alpha=0.3, axis='x')

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'stage13_evaluation.png'), dpi=150, bbox_inches='tight')
    plt.show()


## Stage 14: MODEL COMPARISON


In [ ]:
print("\n" + "="*70)
print("STAGE 14 — MODEL COMPARISON")
print("="*70)

if y_train is not None:
    comparison_rows = [
        {
            'Model'              : 'Baseline (Majority Class)',
            'Accuracy'           : accuracy_score(y_test, y_pred_base),
            'Precision (Flood)'  : precision_score(y_test, y_pred_base, pos_label=1, zero_division=0),
            'Recall (Flood)'     : recall_score(y_test, y_pred_base, pos_label=1, zero_division=0),
            'F1 (Flood)'         : f1_score(y_test, y_pred_base, pos_label=1, zero_division=0),
            'ROC-AUC'            : 0.5000,
            'PR-AUC'             : float(y_test.mean()),
        },
        {
            'Model'              : 'Logistic Regression',
            'Accuracy'           : accuracy_score(y_test, y_pred_lr),
            'Precision (Flood)'  : precision_score(y_test, y_pred_lr, pos_label=1, zero_division=0),
            'Recall (Flood)'     : recall_score(y_test, y_pred_lr, pos_label=1, zero_division=0),
            'F1 (Flood)'         : f1_score(y_test, y_pred_lr, pos_label=1, zero_division=0),
            'ROC-AUC'            : roc_auc_score(y_test, y_prob_lr),
            'PR-AUC'             : average_precision_score(y_test, y_prob_lr),
        },
        {
            'Model'              : 'Random Forest',
            'Accuracy'           : accuracy_score(y_test, y_pred_rf),
            'Precision (Flood)'  : precision_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
            'Recall (Flood)'     : recall_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
            'F1 (Flood)'         : f1_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
            'ROC-AUC'            : roc_auc_score(y_test, y_prob_rf),
            'PR-AUC'             : average_precision_score(y_test, y_prob_rf),
        }
    ]

    comparison_df = pd.DataFrame(comparison_rows).set_index('Model').round(4)
    print(comparison_df.to_string())
    comparison_df.to_csv(os.path.join(OUT_DIR, 'model_comparison.csv'))


## Stage 15: FINAL MODEL SELECTION


In [ ]:
print("\n" + "="*70)
print("STAGE 15 — FINAL MODEL SELECTION")
print("="*70)

if y_train is not None:
    # Select based on ROC-AUC (prioritises flood recall across all thresholds)
    if rf_roc_auc >= lr_roc_auc:
        final_model   = rf_model
        final_name    = 'Random Forest'
        final_prob_fn = lambda X: final_model.predict_proba(X)[:, 1]
        print(f"SELECTED: {final_name}")
        print(f"  ROC-AUC: {rf_roc_auc:.4f} vs LR {lr_roc_auc:.4f}")
        print(f"  Reason: Higher ROC-AUC; captures non-linear terrain-hydrology thresholds.")
    else:
        final_model   = lr_pipeline
        final_name    = 'Logistic Regression'
        final_prob_fn = lambda X: final_model.predict_proba(X)[:, 1]
        print(f"SELECTED: {final_name}")
        print(f"  ROC-AUC: {lr_roc_auc:.4f} vs RF {rf_roc_auc:.4f}")
        print(f"  PR-AUC : {lr_pr_auc:.4f} vs RF {rf_pr_auc:.4f}")
        print(f"  Recall : {recall_score(y_test, y_pred_lr):.4f} vs RF {recall_score(y_test, y_pred_rf):.4f}")
        print(f"  Reason : Higher ROC-AUC, higher PR-AUC, and superior flood recall ({recall_score(y_test, y_pred_lr)*100:.1f}% vs {recall_score(y_test, y_pred_rf)*100:.1f}%),")
        print(f"           minimizing catastrophic false negatives in disaster emergency risk mapping.")
else:
    print("[INFO] Complete modelling stages first.")


## Stage 16: SPATIAL PREDICTION


In [ ]:
print("\n" + "="*70)
print("STAGE 16 — SPATIAL PREDICTION")
print("="*70)

if y_train is not None:
    print(f"Applying {final_name} to full study area grid...")

    pred_mask = ~water_mask.copy()
    for name in FEATURE_NAMES:
        pred_mask &= ~np.isnan(feature_arrays[name])

    pr, pc = np.where(pred_mask)
    X_pred = np.column_stack([feature_arrays[f][pr, pc] for f in FEATURE_NAMES])
    print(f"  Predicting {len(pr):,} pixels...")

    flood_prob = final_prob_fn(X_pred)

    susc_grid = np.full(MASTER_SHAPE, np.nan, dtype=np.float32)
    susc_grid[pr, pc] = flood_prob.astype(np.float32)

    print(f"  Susceptibility: {np.nanmin(flood_prob):.3f} – {np.nanmax(flood_prob):.3f}")
    print(f"  Mean: {np.nanmean(flood_prob):.3f}")

    # Save susceptibility raster
    susc_path = os.path.join(MAP_DIR, 'flood_susceptibility.tif')
    with rasterio.open(dem_proj_path) as ref:
        meta = ref.meta.copy()
        meta.update({'dtype': 'float32', 'nodata': -9999})
    so = susc_grid.copy(); so[np.isnan(susc_grid)] = -9999
    with rasterio.open(susc_path, 'w', **meta) as dst:
        dst.write(so, 1)
    print(f"  Susceptibility raster saved: {susc_path}")


## Stage 17: FLOOD SUSCEPTIBILITY MAP


In [ ]:
print("\n" + "="*70)
print("STAGE 17 — FLOOD SUSCEPTIBILITY MAP")
print("="*70)

RISK_BINS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.001]
RISK_LABELS = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
RISK_COLORS = ['#2ECC71', '#F1C40F', '#E67E22', '#E74C3C', '#8E44AD']

if y_train is not None:
    risk_grid = np.full(MASTER_SHAPE, np.nan, dtype=np.float32)
    for i, (lo, hi) in enumerate(zip(RISK_BINS[:-1], RISK_BINS[1:])):
        risk_grid[(susc_grid >= lo) & (susc_grid < hi)] = i + 1

    print("Flood Susceptibility Class Distribution:")
    n_total_valid = int(np.sum(~np.isnan(risk_grid)))
    for i, label in enumerate(RISK_LABELS):
        cnt = int(np.sum(risk_grid == i+1))
        pct = cnt / max(n_total_valid, 1) * 100
        print(f"  {label:<12}: {cnt:>8,} pixels ({pct:.1f}%)")

    # ── Static susceptibility map ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(
        f"Flood Susceptibility Map — Lokoja Confluence, Kogi State, Nigeria\n"
        f"Model: {final_name}  |  Reference: October 2022  |  Grid: 90m UTM 32N",
        fontsize=13, fontweight='bold'
    )

    # Continuous
    im1 = axes[0].imshow(susc_grid, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(im1, ax=axes[0], label='Flood Susceptibility P(flood)', shrink=0.8)
    axes[0].set_title("Continuous Susceptibility Index", fontweight='bold')
    axes[0].set_xlabel("West → East"); axes[0].set_ylabel("South → North")

    # Categorical
    cmap_r = mcolors.ListedColormap(RISK_COLORS)
    norm_r = mcolors.BoundaryNorm([0.5,1.5,2.5,3.5,4.5,5.5], cmap_r.N)
    im2 = axes[1].imshow(risk_grid, cmap=cmap_r, norm=norm_r, aspect='auto')
    cbar = plt.colorbar(im2, ax=axes[1], ticks=[1,2,3,4,5], shrink=0.8)
    cbar.ax.set_yticklabels(RISK_LABELS)
    cbar.set_label("Risk Category")
    axes[1].set_title("Categorised Risk (5 Tiers)", fontweight='bold')
    axes[1].set_xlabel("West → East")

    plt.tight_layout()
    map_path = os.path.join(MAP_DIR, 'flood_susceptibility_map.png')
    plt.savefig(map_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"\nMap saved: {map_path}")

    # ── Interactive Folium map ─────────────────────────────────────────────
    inv_t = Transformer.from_crs(CRS_PROJ, CRS_GEO, always_xy=True)
    centre_lat = (BBOX_NORTH + BBOX_SOUTH) / 2
    centre_lon = (BBOX_WEST + BBOX_EAST) / 2

    m = folium.Map(location=[centre_lat, centre_lon], zoom_start=10,
                   tiles='OpenStreetMap')

    # High/Very High risk points heatmap
    high_mask = (susc_grid >= 0.6) & ~np.isnan(susc_grid)
    hr, hc = np.where(high_mask)
    if len(hr) > 0:
        sample_idx = np.random.choice(len(hr), min(5000, len(hr)), replace=False)
        xc = x_min + (hc[sample_idx] + 0.5) * RESOLUTION_M
        yc = y_max - (hr[sample_idx] + 0.5) * RESOLUTION_M
        lons, lats = inv_t.transform(xc, yc)
        probs = susc_grid[hr[sample_idx], hc[sample_idx]]
        heat_data = [[float(lat), float(lon), float(p)]
                     for lat, lon, p in zip(lats, lons, probs)
                     if -90 <= lat <= 90 and -180 <= lon <= 180]
        HeatMap(heat_data, min_opacity=0.3, radius=10, blur=8,
                gradient={'0.4':'#F1C40F','0.65':'#E67E22',
                          '0.8':'#E74C3C','1.0':'#8E44AD'}).add_to(m)

    folium.Rectangle(
        bounds=[[BBOX_SOUTH, BBOX_WEST], [BBOX_NORTH, BBOX_EAST]],
        color='#2C3E50', weight=2, fill=False,
        tooltip='Lokoja Confluence Study Area'
    ).add_to(m)

    folium.LayerControl().add_to(m)
    html_path = os.path.join(MAP_DIR, 'flood_susceptibility_interactive.html')
    m.save(html_path)
    print(f"Interactive map saved: {html_path}")


## Stage 18: FINDINGS, LIMITATIONS, DECISION LOG & CONCLUSION


In [ ]:
print("\n" + "="*70)
print("STAGE 18 — FINDINGS, LIMITATIONS & CONCLUSION")
print("="*70)

# ── Documented Limitations ────────────────────────────────────────────────
limitations = [
    {'ID':'LIM-01','Category':'Data',
     'Issue':'Single event (October 2022 only)',
     'Impact':'Model may not generalise to other flood types or severities',
     'Mitigation':'Acknowledged; multi-event training recommended as future work'},
    {'ID':'LIM-02','Category':'Data',
     'Issue':'CHIRPS at 5.3km downscaled to 90m via bilinear interpolation',
     'Impact':'Within-cell rainfall variation is not captured; smoothing introduced',
     'Mitigation':'Acknowledged; rainfall gradient in study area is moderate'},
    {'ID':'LIM-03','Category':'Spatial',
     'Issue':'Spatial block split (quadrant-based) is not full spatial k-fold CV',
     'Impact':'Metrics may be slightly optimistic at quadrant boundaries',
     'Mitigation':'Limitation documented; rigorous spatial CV recommended for future work'},
    {'ID':'LIM-04','Category':'Model',
     'Issue':'Class imbalance (far more dry than flooded pixels, 98.4% dry in test holdout)',
     'Impact':'Model accuracy is misleading without majority baseline reference',
     'Mitigation':'class_weight="balanced" applied; evaluated using ROC-AUC and PR-AUC'},
    {'ID':'LIM-05','Category':'Data',
     'Issue':'No antecedent soil moisture data',
     'Impact':'Pre-event soil saturation state is unknown but highly relevant',
     'Mitigation':'Not feasible within scope; documented gap'},
    {'ID':'LIM-06','Category':'Interpretation',
     'Issue':'Multivariable coefficients and feature importance reflect conditional association, not causation',
     'Impact':'Positive slope/dist_river coefficients reflect suppressor effect from conditioning on elevation; negative rainfall reflects fluvial hydrology',
     'Mitigation':'Language throughout uses "associated with" not "causes"; conditioning and suppressor mechanics documented'},
    {'ID':'LIM-07','Category':'Data',
     'Issue':'UNOSAT VIIRS optical/radiometer flood delineation accuracy',
     'Impact':'Native 375m VIIRS resolution resampled to 90m; cloud obstruction mitigated via temporal composite',
     'Mitigation':'UNOSAT FL20221019NGA is the verified official product; limitations and compositing noted'},
    {'ID':'LIM-08','Category':'Spatial',
     'Issue':'Spatial Domain Shift in Test Quadrant',
     'Impact':'SE test quadrant is +100.7m higher mean elevation with only 1.57% flood prevalence',
     'Mitigation':'Documented as real-world generalization test rather than artificially rebalancing holdout'},
    {'ID':'LIM-09','Category':'Validation',
     'Issue':'Model selection evaluated on holdout partition',
     'Impact':'Holdout functioned as a validation partition for model comparison; performance may have slight selection optimism',
     'Mitigation':'Acknowledged as scope limitation; nested spatial cross-validation recommended for future work'},
    {'ID':'LIM-10','Category':'Model',
     'Issue':'Predicted probabilities are uncalibrated relative susceptibility indices',
     'Impact':'class_weight="balanced" shifts probability scale upward; values represent monotonic relative risk rankings, not true empirical frequencies',
     'Mitigation':'FSI explicitly interpreted as relative susceptibility index, not calibrated probability'},
]

print("\nDocumented Limitations:")
for lim in limitations:
    print(f"\n  {lim['ID']} [{lim['Category']}]")
    print(f"    Issue      : {lim['Issue']}")
    print(f"    Impact     : {lim['Impact']}")
    print(f"    Mitigation : {lim['Mitigation']}")

lim_df = pd.DataFrame(limitations)
lim_df.to_csv(os.path.join(OUT_DIR, 'limitations.csv'), index=False)


# ── Decision Log ───────────────────────────────────────────────────────────
decision_log = [
    {'ID':'DEC-001','Stage':'Feasibility',
     'Decision':'Binary classification + probability output as FSI',
     'Alternatives':'Regression, Clustering',
     'Reason':'UNOSAT provides clean binary ground-truth; probability enables continuous map',
     'Status':'Final'},
    {'ID':'DEC-002','Stage':'Study Area',
     'Decision':'Lokoja Confluence, Kogi State, Nigeria',
     'Alternatives':'Lagos, Anambra, US FEMA basin',
     'Reason':'Oct 2022 disaster with verified UNOSAT labels; strong terrain variation; confluent rivers',
     'Status':'Final'},
    {'ID':'DEC-003','Stage':'Elevation',
     'Decision':'SRTM 30m (OpenTopography)',
     'Alternatives':'ALOS PALSAR 12.5m, ASTER 30m',
     'Reason':'Free, globally available, no authentication for small areas',
     'Status':'Final'},
    {'ID':'DEC-004','Stage':'Rainfall',
     'Decision':'CHIRPS v2.0 monthly (Sep+Oct 2022)',
     'Alternatives':'GPM IMERG, ERA5-Land',
     'Reason':'Finer resolution than GPM (5km vs 11km); no API auth required',
     'Status':'Final'},
    {'ID':'DEC-005','Stage':'River Network',
     'Decision':'HydroRIVERS Africa',
     'Alternatives':'DEM-derived streams only',
     'Reason':'Topologically verified; avoids broken channels from coarse DEM extraction',
     'Status':'Final'},
    {'ID':'DEC-006','Stage':'Land Cover',
     'Decision':'ESA WorldCover 2021 (10m)',
     'Alternatives':'Raw Sentinel-2 classification',
     'Reason':'Pre-classified authoritative product; avoids raw imagery classification complexity',
     'Status':'Final'},
    {'ID':'DEC-007','Stage':'Flood Labels',
     'Decision':'UNOSAT FL20221019NGA VIIRS flood extent (Product 3427)',
     'Alternatives':'Sentinel-1 SAR (unverified local footprint), Government records (unavailable)',
     'Reason':'Official verified UNOSAT multi-day composite (1-25 Oct 2022); 375m VIIRS sensor; open access',
     'Status':'Final'},
    {'ID':'DEC-008','Stage':'CRS',
     'Decision':'UTM Zone 32N (EPSG:32632)',
     'Alternatives':'WGS84 (EPSG:4326)',
     'Reason':'Metric CRS required for slope, Euclidean distance, and area accuracy',
     'Status':'Final'},
    {'ID':'DEC-009','Stage':'Resolution',
     'Decision':'90m master grid',
     'Alternatives':'30m (too large for Colab RAM), 250m (too coarse)',
     'Reason':'Balances spatial detail with computation feasibility in Google Colab',
     'Status':'Final'},
    {'ID':'DEC-010','Stage':'Validation',
     'Decision':'Spatial block split (2×2 quadrant; SE = test)',
     'Alternatives':'Random split (spatial leakage), spatial k-fold (too complex)',
     'Reason':'Random split causes data leakage; k-fold too complex for beginner scope',
     'Status':'Final'},
    {'ID':'DEC-011','Stage':'Models',
     'Decision':'Logistic Regression + Random Forest',
     'Alternatives':'XGBoost, SVM, Neural Network',
     'Reason':'Simplest interpretable pair; LR is linear baseline; RF handles non-linear thresholds',
     'Status':'Final'},
    {'ID':'DEC-012','Stage':'Final Model',
     'Decision':'Selected by highest ROC-AUC after Stage 14 evaluation',
     'Alternatives':'N/A',
     'Reason':'Evidence-based; prioritises flood recall across all probability thresholds',
     'Status':'Completed in Stage 15'},
    {'ID':'DEC-013','Stage':'Evaluation Metrics',
     'Decision':'Mandatory ROC-AUC and PR-AUC reporting',
     'Alternatives':'Accuracy alone (misleading under 98.4% dry imbalance)',
     'Reason':'PR-AUC directly reflects positive-class precision and recall under extreme rarity',
     'Status':'Final'},
    {'ID':'DEC-014','Stage':'Baseline Benchmark',
     'Decision':'Naive majority-class classifier included in evaluation',
     'Alternatives':'No baseline',
     'Reason':'Essential to demonstrate performance gain over trivial all-dry prediction',
     'Status':'Final'},
]

dec_df = pd.DataFrame(decision_log)
print("\n\nDecision Log:")
print(dec_df[['ID','Stage','Decision','Status']].to_string(index=False))
dec_df.to_csv(os.path.join(OUT_DIR, 'decision_log.csv'), index=False)


# ── Save project metadata ───────────────────────────────────────────────────
meta_out = {
    'project'         : 'AI-Powered Flood Risk Prediction',
    'study_area'      : 'Lokoja Confluence, Kogi State, Nigeria',
    'reference_event' : 'October 2022 Nigeria Flood',
    'crs'             : MASTER_CRS,
    'resolution_m'    : RESOLUTION_M,
    'bbox'            : {'west':BBOX_WEST,'east':BBOX_EAST,'south':BBOX_SOUTH,'north':BBOX_NORTH},
    'features'        : FEATURE_NAMES,
    'random_state'    : RANDOM_STATE,
}

if y_train is not None:
    meta_out.update({
        'final_model'  : final_name,
        'train_samples': int(len(df_train)),
        'test_samples' : int(len(df_test)),
        'roc_auc_lr'   : float(lr_roc_auc),
        'roc_auc_rf'   : float(rf_roc_auc),
    })

with open(os.path.join(OUT_DIR, 'project_metadata.json'), 'w') as f:
    json.dump(meta_out, f, indent=2)


print("\n" + "="*70)
print("CONCLUSION")
print("="*70)
print("""
This notebook demonstrates a complete, reproducible flood susceptibility
workflow for the Lokoja Confluence, Kogi State, Nigeria — centred on the
October 2022 peak flood event.

WHAT WAS ACHIEVED:
  • Integrated 5 authoritative datasets into a common 90m UTM spatial grid
  • Derived 5 hydrologically-meaningful features
  • Applied spatial block splitting to reduce data leakage
  • Trained and compared two models (Logistic Regression vs Random Forest)
  • Generated continuous Flood Susceptibility Index and 5-tier risk map
  • Documented all 12 major decisions and 7 key limitations

CRITICAL NOTE:
  This model estimates FLOOD SUSCEPTIBILITY based on October 2022 conditions.
  It does NOT predict future floods with certainty. Results are a relative
  spatial indicator and should not replace professional flood hazard assessments.
""")


print("\n" + "="*70)
print("REFERENCES")
print("="*70)
references = [
    "1. Funk, C. et al. (2015). The climate hazards infrared precipitation with stations."
    " Scientific Data, 2, 150066. https://doi.org/10.1038/sdata.2015.66",
    "2. NASA SRTM (2013). Shuttle Radar Topography Mission Global."
    " https://doi.org/10.5066/F7PR7TFT",
    "3. Lehner, B. et al. (2008). New global hydrography derived from spaceborne elevation data."
    " Eos, 89(10), 93–94.",
    "4. Zanaga, D. et al. (2022). ESA WorldCover 10m 2021 v200. Zenodo."
    " https://doi.org/10.5281/zenodo.7254221",
    "5. UNOSAT (2022). Satellite detected water extent over Kogi State, Nigeria."
    " Flood event: 14 October 2022. UNITAR-UNOSAT.",
    "6. Breiman, L. (2001). Random Forests. Machine Learning, 45(1), 5–32.",
    "7. Cox, D.R. (1958). The regression analysis of binary sequences."
    " J. Royal Statistical Society B, 20(2), 215–242.",
]
for ref in references:
    print(f"  {ref}")

print("\n" + "="*70)
print("PIPELINE COMPLETE — All outputs saved to:", OUT_DIR)
print("="*70)
